# III. Data Cleaning & Preparation

Dựa trên mục tiêu nghiên cứu và phạm vi đã xác định, bộ dữ liệu được lọc và xử lý theo các tiêu chí:
- Quốc gia báo cáo: Việt Nam
- Luồng thương mại: Xuất khẩu
- Giai đoạn: 2019–2023
- Phân loại hàng hóa: HS 2 chữ số
- Đối tác thương mại: Tổng thế giới

In [ ]:
print("Raw shape:", df.shape)
df.head()

## 3.1. Xác định và kiểm tra phạm vi dữ liệu nghiên cứu 

In [ ]:
# Lọc lại df_scope với partnerISO = W00
mask = (
    df["reporterISO"].isin(["VNM", "VN"]) &
    df["flowDesc"].str.contains("Export", case=False, na=False) &
    df["refYear"].between(2019, 2023) &
    (df["partnerISO"] == "W00")  # chỉ lấy tổng thế giới
)

df_scope = df.loc[mask].copy()  

print("After scope filtering:", df_scope.shape)
df_scope[["refYear", "reporterISO", "flowDesc","partnerISO"]].drop_duplicates().head(10)

Dữ liệu được lọc theo phạm vi nghiên cứu gồm: Việt Nam (reporterISO = VNM), luồng xuất khẩu (Export), đối tác thương mại: Tổng thế giới (partnerISO = W00) và giai đoạn 2019–2023.  
Kết quả kiểm tra cho thấy dữ liệu sau lọc chỉ chứa các năm trong phạm vi nghiên cứu và đúng luồng giao dịch xuất khẩu, đảm bảo tính nhất quán trước khi xử lý sâu hơn.


## 3.2. Kiểm tra độ đầy đủ của các cột giá trị 

In [ ]:
value_cols = ["primaryValue", "fobvalue", "cifvalue"]
missing_report = df_scope[value_cols].isna().sum().to_frame("missing_count")
missing_report["missing_rate"] = missing_report["missing_count"] / len(df_scope)
missing_report.sort_values("missing_rate", ascending=False)

Kết quả kiểm tra mức độ thiếu dữ liệu cho thấy sự khác biệt rõ rệt giữa các biến giá trị giao dịch.  
Trong khi biến `primaryValue` và `fobvalue` có dữ liệu đầy đủ cho toàn bộ các bản ghi, biến `cifvalue` lại thiếu dữ liệu ở hơn 60% số giao dịch.

Do tỷ lệ thiếu dữ liệu của `cifvalue` ở mức cao, việc sử dụng biến này trong phân tích tổng thể có thể dẫn đến sai lệch và làm giảm tính đại diện của kết quả.  
Vì vậy, trong phạm vi nghiên cứu này, `primaryValue` được lựa chọn làm biến đại diện cho giá trị xuất khẩu nhằm đảm bảo tính nhất quán và độ tin cậy của các phân tích tiếp theo.


## 3.3. Lựa chọn các biến phục vụ phân tích

In [ ]:
keep_cols = [
    "refYear",       # Năm
    "cmdCode",       # Mã hàng (HS)
    "cmdDesc",       # Mô tả hàng hóa
    "aggrLevel",     # Mức độ tổng hợp 
    "primaryValue",  # Giá trị giao dịch 
]
keep_cols = [c for c in keep_cols if c in df_scope.columns]  # tránh lỗi nếu thiếu cột

df_prepared = df_scope[keep_cols].copy()
print("Selected shape:", df_prepared.shape)
df_prepared.head()

Sau khi xác định phạm vi nghiên cứu và kiểm tra mức độ đầy đủ của các biến giá trị, bộ dữ liệu được tiếp tục tinh giản bằng cách lựa chọn các biến phục vụ trực tiếp cho mục tiêu phân tích. Cụ thể, các biến được giữ lại bao gồm thông tin về thời gian giao dịch, mã và mô tả hàng hóa, mức độ tổng hợp theo HS, cùng với giá trị giao dịch xuất khẩu. Việc lựa chọn này giúp giảm độ phức tạp của dữ liệu, đồng thời vẫn đảm bảo đầy đủ thông tin cần thiết để phân tích xu hướng, cơ cấu ngành hàng và mức độ tập trung xuất khẩu.

In [ ]:
df_prepared.rename(columns={
    "refYear": "Year",
    "cmdCode": "HS_Code",
    "cmdDesc": "HS_Desc",
    "primaryValue": "Trade_Value"
}, inplace=True)

df_prepared.info()

Trong phạm vi nghiên cứu, dữ liệu được tinh giản chỉ giữ lại các trường cần thiết để phân tích theo năm và theo nhóm ngành hàng: năm (Year), mã HS (HS_Code), mô tả ngành (HS_Desc) và giá trị giao dịch (Trade_Value).  
Việc đổi tên cột giúp tăng tính nhất quán và dễ đọc trong quá trình phân tích và trực quan hóa, đặc biệt khi triển khai trên Power BI.


## 3.4. Chuẩn hóa HS về HS 2-digit

Dữ liệu gốc đã ở cấp HS 2 chữ số, bước xử lý này chỉ nhằm chuẩn hóa cách biểu diễn mã ngành về dạng 2 ký tự, ví dụ 1 thành 01.

In [ ]:
def format_hs2_code(x):
    if pd.isna(x):
        return np.nan

    s = str(int(x))
    return s.zfill(2)


df_prepared["HS2"] = df_prepared["HS_Code"].apply(format_hs2_code)

hs2_summary = (
    df_prepared["HS2"]
    .value_counts()
    .rename("Số bản ghi")
    .to_frame()
)

hs2_summary["Tỷ trọng (%)"] = (
    hs2_summary["Số bản ghi"] / hs2_summary["Số bản ghi"].sum() * 100
)

hs2_summary = hs2_summary.reset_index().rename(columns={"index": "HS 2-digit"})

hs2_summary.head(10).style.format({
    "Tỷ trọng (%)": "{:.2f}%"
})

Kết quả trên cho thấy dữ liệu xuất khẩu của Việt Nam được phân bố trên nhiều nhóm ngành hàng khác nhau theo phân loại HS 2 chữ số.  

## 3.5. Chuẩn hóa mô tả ngành HS
Trong dữ liệu gốc, một số mã HS có nhiều mô tả khác nhau do khác biệt về cách diễn đạt như bạn có thể thấy dưới đây 

In [ ]:
desc_check = (
    df_prepared.groupby("HS2")["HS_Desc"]
    .nunique()
    .reset_index()
)

desc_check[desc_check["HS_Desc"] > 1]

In [ ]:
hs_desc_multi = (
    df_prepared.groupby("HS2")["HS_Desc"]
    .agg(lambda x: sorted(set(x.dropna())))
    .reset_index()
)

hs_desc_multi["Desc_Count"] = hs_desc_multi["HS_Desc"].apply(len)
hs_desc_multi = hs_desc_multi[hs_desc_multi["Desc_Count"] > 1].copy()

hs_desc_multi = hs_desc_multi.explode("HS_Desc").sort_values(["HS2", "HS_Desc"]).reset_index(drop=True)

hs_desc_multi

Để đảm bảo tính nhất quán khi hiển thị và phân tích, mỗi mã HS được gán một mô tả đại diện duy nhất.

In [ ]:
hs_desc_standard = (
    df_prepared
    .groupby("HS2")["HS_Desc"]
    .last()
    .reset_index()
)

df_prepared = df_prepared.drop(columns="HS_Desc").merge(
    hs_desc_standard,
    on="HS2",
    how="left"
)

In [ ]:
df_prepared.groupby("HS2")["HS_Desc"].nunique().value_counts()

In [ ]:
df_prepared[df_prepared["HS2"].isin(["15","16","24","84","88"])][["HS2","HS_Desc"]].drop_duplicates()

## 3.6. Xử lý giá trị & kiểu dữ liệu
 - Mục tiêu: đảm bảo Trade_Value là số và không âm.

In [ ]:
df_prepared["Trade_Value"] = pd.to_numeric(df_prepared["Trade_Value"], errors="coerce")

before = len(df_prepared)
df_prepared = df_prepared.dropna(subset=["Trade_Value"])
after = len(df_prepared)
print(f"Dropped rows with missing Trade_Value: {before - after}")

neg_count = (df_prepared["Trade_Value"] < 0).sum()
print("Negative Trade_Value count:", neg_count)

df_prepared = df_prepared[df_prepared["Trade_Value"] >= 0].copy()

df_prepared[["Year", "HS2", "Trade_Value"]].describe(include="all")

Sau khi chuẩn hóa kiểu dữ liệu, biến `Trade_Value` được chuyển về dạng số để đảm bảo có thể sử dụng trong các phép tổng hợp và trực quan hóa. Kết quả cho thấy dữ liệu không có bản ghi bị thiếu `Trade_Value` và không tồn tại giá trị âm. Điều này giúp đảm bảo độ tin cậy khi sử dụng `Trade_Value` làm biến đại diện cho quy mô xuất khẩu trong các phân tích tiếp theo.

Thống kê mô tả cũng cho thấy dữ liệu bao gồm nhiều nhóm ngành HS 2 chữ số (97 nhóm), trong đó có một số nhóm xuất hiện thường xuyên hơn. Đặc biệt, sự chênh lệch lớn giữa giá trị nhỏ nhất và lớn nhất phản ánh đặc điểm phân bố lệch của dữ liệu thương mại, do một số ngành có quy mô xuất khẩu vượt trội so với phần còn lại.


## 3.7. Tổng hợp dữ liệu theo năm và nhóm ngành hàng

In [ ]:
df_grouped = (
    df_prepared
    .groupby(["Year", "HS2"], as_index=False)
    .agg(
        Total_Export_Value=("Trade_Value", "sum"),
        Record_Count=("Trade_Value", "size")
    )
)

df_grouped = df_grouped.sort_values(["Year", "Total_Export_Value"], ascending=[True, False])

print("Aggregated shape:", df_grouped.shape)
df_grouped.head(10)


Sau khi tổng hợp dữ liệu theo năm và nhóm ngành hàng (HS 2 chữ số), bảng dữ liệu thu được gồm 484 dòng và 4 cột.  Mỗi dòng đại diện cho một nhóm ngành cụ thể trong một năm, kèm theo tổng giá trị xuất khẩu và số lượng bản ghi gốc được cộng lại để tạo nên giá trị này. Cấu trúc dữ liệu này giúp chuyển dữ liệu từ mức chi tiết sang mức tổng hợp, phù hợp cho việc phân tích xu hướng, cơ cấu ngành và trực quan hóa trong các bước tiếp theo.

## 3.8. Tạo biến “Industry_Group” để phân tích theo nhóm ngành lớn
Để tăng khả năng diễn giải theo góc nhìn nghiệp vụ, các mã HS2 được ánh xạ sang nhóm ngành lớn (Industry_Group).

In [ ]:
industry_map = {
    'Nông - Thủy sản':               ['01','02','03','04','05','06','07','08','09','10','11','12','13','14'],
    'Thực phẩm chế biến':            ['15','16','17','18','19','20','21','22','23','24'],
    'Khoáng sản & Năng lượng':       ['25','26','27'],
    'Hóa chất & Nhựa':               ['28','29','30','31','32','33','34','35','36','37','38','39','40'],
    'Dệt may - Da giày':             ['41','42','43','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66','67'],
    'Gỗ - Giấy - Nội thất':          ['44','47','48','94'],
    'Đá - Thủy tinh - VL xây dựng':  ['68','69','70'],
    'Kim loại':                      ['71','72','73','74','75','76','78','79','80','81','82','83'],
    'Máy móc - Điện tử':             ['84','85'],
    'Phương tiện vận tải':           ['86','87','88','89'],
    'Dụng cụ & Thiết bị chuyên dụng':['90','91','92'],
    'Hàng tiêu dùng':                ['45','46','49','95','96'],
}

def assign_industry(hs2):
    hs2 = str(hs2).zfill(2)
    for group, codes in industry_map.items():
        if hs2 in codes:
            return group
    return "Khác"

df_grouped["Industry_Group"] = df_grouped["HS2"].apply(assign_industry)

# check
df_grouped["Industry_Group"].value_counts()

In [ ]:
# Kiểm tra tỷ trọng nhóm "Khác" — các mã HS không được phân loại
khac = df_grouped[df_grouped["Industry_Group"] == "Khác"]

tong_xk = df_grouped["Total_Export_Value"].sum()
tong_khac = khac["Total_Export_Value"].sum()
ty_trong_khac = tong_khac / tong_xk * 100

print(f"Số mã HS2 thuộc nhóm 'Khác': {khac['HS2'].nunique()}")
print(f"Các mã HS2 đó là: {sorted(khac['HS2'].unique().tolist())}")
print(f"Tổng giá trị xuất khẩu nhóm 'Khác': {tong_khac/1e6:,.1f} triệu USD")
print(f"Tỷ trọng trong tổng xuất khẩu: {ty_trong_khac:.3f}%")

Bước này giúp giảm độ chi tiết khi phân tích, phù hợp với mục tiêu đánh giá cơ cấu và mức độ ổn định theo ngành trong bối cảnh nghiệp vụ ngân hàng. Ngoài ra ta có thể thấy Nhóm 'Khác' chiếm dưới 2%, ảnh hưởng không đáng kể đến dữ liệu.

In [ ]:
hs_list = (
    df_prepared[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)

hs_list.head(20)

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)        
pd.set_option("display.max_columns", None)     
pd.set_option("display.max_colwidth", None)    
pd.set_option("display.width", 1000)           

hs_list = (
    df_prepared[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)

display(hs_list)

Danh mục HS2 – HS_Desc được tách riêng nhằm phục vụ tra cứu và hiển thị nhãn ngành trong dashboard Power BI. Toàn bộ danh mục có thể được đưa vào phần phụ lục để đảm bảo tính minh bạch và hỗ trợ người đọc hiểu rõ ý nghĩa các mã HS sử dụng trong nghiên cứu.

In [ ]:
#check
print("df_grouped:", df_grouped.shape)
print("hs_list:", hs_list.shape)

df_grouped.head()